# Module 3 Exercise: Copy task — vanilla RNN vs. LSTM, with gradient-flow diagnostics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/03-rnn-lstm/exercise_starter.ipynb)

Module page: [Module 3: RNNs + LSTMs](https://nsteve2407.github.io/llm-transformers-course/modules/03-rnn-lstm/)

**Part A**: build a synthetic copy-task data generator (a standard long-range-dependency benchmark).

**Part B**: implement a vanilla RNN cell and an LSTM cell manually, using raw tensor ops only (no `nn.RNN`/`nn.RNNCell`/`nn.LSTM`/`nn.LSTMCell`).

**Part C**: train both on the copy task across a sweep of delay lengths, recording final accuracy per `(model, delay)` pair.

**Part D**: gradient-flow diagnostic — backprop through a fixed long delay and record `||dL/dh_t||` (and, for LSTM, `||dL/dc_t||`) at every timestep.

**Part E**: plot accuracy vs. delay, and the gradient-flow diagnostic, for both models.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Task design and hyperparameters (documented judgment calls)

We use the classic **copy task** (Hochreiter & Schmidhuber; also used by Arjovsky et al. 2015, *"Unitary
Evolution Recurrent Neural Networks"*, as a standard synthetic benchmark for long-range dependencies). Each
input sequence has the structure:

```
[ L data symbols ] [ T-1 blanks ] [ delimiter ] [ L blanks ]
```

The model reads the `L` data symbols, must carry them across `T-1` blank steps, and after seeing the
delimiter must reproduce the original `L` symbols (in order) during the final `L` output timesteps. Total
sequence length is `2L + T`, so larger `T` directly means a longer gap the model's hidden state must bridge
-- this is exactly the vanishing-gradient stress test.

**Judgment calls made here (small vocab, short `L`, moderate hidden size) -- documented rather than
mandated by the brief:**
- `NUM_SYMBOLS = 8` data symbols (ids 1-8), plus a blank id (0) and a delimiter id (9) -> vocab size 10.
- `COPY_LEN = 3`: a short symbol sequence to memorize, so the *delay* `T` is the sole variable under test
  (not the amount of information to store).
- Delay sweep `T in {5, 20, 50, 100}` (full run) / `{2, 4}` (`SMOKE_TEST`), per the task brief.
- `HIDDEN_SIZE = 64` for **both** cell types, matching hidden-state (memory) dimensionality. LSTM still ends
  up with ~4x the parameters of the RNN because it has 4 gates instead of 1 -- that asymmetry is inherent to
  the architectures being compared, not something we equalize away.
- The LSTM's forget-gate bias is initialized to `+1` (a well-known trick, e.g. Jozefowicz et al. 2015) so
  gates start closer to "remember by default," which is itself part of why LSTMs handle long delays better.
- Training uses gradient clipping (max norm 5.0) to keep the vanilla RNN's occasional exploding gradients
  from producing `NaN` losses and derailing the accuracy sweep. Clipping addresses *exploding* gradients
  only -- it does nothing for the *vanishing*-gradient problem this notebook is built to expose.
- The gradient-flow diagnostic (Part D) uses **freshly initialized, untrained** models rather than the
  models trained in Part C, so the measured gradient norms reflect architecture/initialization, not whatever
  the optimizer already reshaped during training.

In [ ]:
NUM_SYMBOLS = 8            # data symbols use ids 1..NUM_SYMBOLS
BLANK_ID = 0
DELIM_ID = NUM_SYMBOLS + 1  # 9
VOCAB_SIZE = NUM_SYMBOLS + 2  # 10 (blank, 8 data symbols, delimiter)
COPY_LEN = 3                # length L of the symbol sequence to memorize/reproduce

D_MODEL = 16
HIDDEN_SIZE = 64

DELAY_VALUES = [2, 4] if SMOKE_TEST else [5, 20, 50, 100]
GRAD_DIAG_T = max(DELAY_VALUES)  # fixed long delay used for the gradient-flow diagnostic

TRAIN_STEPS = 20 if SMOKE_TEST else 500
TRAIN_BATCH_SIZE = 32 if SMOKE_TEST else 128
EVAL_BATCH_SIZE = 64 if SMOKE_TEST else 512

print(f"VOCAB_SIZE={VOCAB_SIZE}, COPY_LEN={COPY_LEN}, DELAY_VALUES={DELAY_VALUES}, GRAD_DIAG_T={GRAD_DIAG_T}")

## Part A: synthetic copy-task data generator

In [ ]:
def make_copy_batch(batch_size, T, L=COPY_LEN, num_symbols=NUM_SYMBOLS, device="cpu"):
    """Build a batch of copy-task sequences with delay length T.

    Layout per sequence: [L random data symbols] [T-1 blanks] [delimiter] [L blanks].
    The target is just the original L data symbols -- the model must reproduce them,
    in order, during the final L timesteps (right after the delimiter).
    """
    to_copy = torch.randint(1, num_symbols + 1, (batch_size, L))
    mid_blanks = torch.zeros(batch_size, T - 1, dtype=torch.long)
    delim = torch.full((batch_size, 1), DELIM_ID, dtype=torch.long)
    out_blanks = torch.zeros(batch_size, L, dtype=torch.long)
    inputs = torch.cat([to_copy, mid_blanks, delim, out_blanks], dim=1)
    return inputs.to(device), to_copy.to(device)

In [ ]:
# Sanity check: print one example sequence for a short delay.
x_demo, y_demo = make_copy_batch(1, T=5)
print("input sequence: ", x_demo[0].tolist())
print("target (last L symbols must match this):", y_demo[0].tolist())
print("sequence length:", x_demo.shape[1], "= 2*COPY_LEN + T =", 2 * COPY_LEN + 5)

## Part B: manual vanilla RNN cell and manual LSTM cell

Both cells operate on one timestep at a time (`x_t`, previous state(s)) using only raw tensor ops -- no
`nn.RNN`/`nn.RNNCell`/`nn.LSTM`/`nn.LSTMCell`. `CopyModel` below loops them over the sequence.

In [ ]:
class ManualRNNCell(nn.Module):
    """Vanilla RNN cell: h_t = tanh(W_ih @ x_t + W_hh @ h_{t-1} + b)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(input_size, hidden_size) * input_size ** -0.5)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * hidden_size ** -0.5)
        self.b = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t, h_prev):
        # TODO: h_t = tanh(x_t @ self.W_ih + h_prev @ self.W_hh + self.b). Return h_t.
        raise NotImplementedError("TODO: implement ManualRNNCell.forward")

In [ ]:
class ManualLSTMCell(nn.Module):
    """LSTM cell with the four standard gates (input/forget/candidate/output) and the
    additive cell-state update c_t = f_t * c_{t-1} + i_t * g_t, h_t = o_t * tanh(c_t).

    All four gates' pre-activations are computed with one matmul each against W_ih/W_hh
    (concatenated along the output dim) and then split with `.chunk(4, dim=1)`.
    """

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_ih = nn.Parameter(torch.randn(input_size, 4 * hidden_size) * input_size ** -0.5)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, 4 * hidden_size) * hidden_size ** -0.5)
        self.b = nn.Parameter(torch.zeros(4 * hidden_size))
        with torch.no_grad():
            # Forget-gate bias init to +1: start closer to "remember by default" (Jozefowicz et al. 2015).
            self.b[hidden_size:2 * hidden_size].fill_(1.0)

    def forward(self, x_t, h_prev, c_prev):
        # TODO: compute gates = x_t @ self.W_ih + h_prev @ self.W_hh + self.b, split into
        # i_pre/f_pre/g_pre/o_pre = gates.chunk(4, dim=1), apply sigmoid to i/f/o and tanh to g,
        # then c_t = f_t * c_prev + i_t * g_t (additive update) and h_t = o_t * tanh(c_t).
        raise NotImplementedError("TODO: implement ManualLSTMCell.forward (4 gates + additive cell update)")

In [ ]:
class CopyModel(nn.Module):
    """Embedding -> manual recurrent cell (looped over time) -> per-timestep readout."""

    def __init__(self, vocab_size, d_model, hidden_size, cell_type):
        super().__init__()
        assert cell_type in ("rnn", "lstm")
        self.cell_type = cell_type
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.cell = ManualRNNCell(d_model, hidden_size) if cell_type == "rnn" else ManualLSTMCell(d_model, hidden_size)
        self.readout = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, return_states=False):
        batch_size, seq_len = x.shape
        emb = self.embedding(x)  # (batch, seq_len, d_model)
        h_t = torch.zeros(batch_size, self.hidden_size, device=x.device)
        c_t = torch.zeros(batch_size, self.hidden_size, device=x.device) if self.cell_type == "lstm" else None

        hidden_states, cell_states, logits = [], [], []
        for t in range(seq_len):
            x_t = emb[:, t, :]
            if self.cell_type == "rnn":
                h_t = self.cell(x_t, h_t)
            else:
                h_t, c_t = self.cell(x_t, h_t, c_t)
            if return_states:
                h_t.retain_grad()
                hidden_states.append(h_t)
                if c_t is not None:
                    c_t.retain_grad()
                    cell_states.append(c_t)
            logits.append(self.readout(h_t))

        logits = torch.stack(logits, dim=1)  # (batch, seq_len, vocab_size)
        if return_states:
            return logits, hidden_states, cell_states
        return logits

In [ ]:
# Parameter-count sanity check: same hidden size, LSTM naturally has ~4x the params (4 gates vs. 1).
rnn_probe = CopyModel(VOCAB_SIZE, D_MODEL, HIDDEN_SIZE, cell_type="rnn")
lstm_probe = CopyModel(VOCAB_SIZE, D_MODEL, HIDDEN_SIZE, cell_type="lstm")
rnn_params = sum(p.numel() for p in rnn_probe.parameters())
lstm_params = sum(p.numel() for p in lstm_probe.parameters())
print(f"RNN params:  {rnn_params}")
print(f"LSTM params: {lstm_params}  ({lstm_params / rnn_params:.2f}x the RNN)")

## Part C: train both models across the delay sweep

For each `(cell_type, T)` pair we train a fresh model on freshly sampled batches (the task is synthetic and
effectively infinite, so we just draw new batches each step rather than using a fixed dataset), then
evaluate copy accuracy (per-symbol) on a held-out batch.

In [ ]:
def copy_task_loss_and_acc(logits, targets, L):
    """Loss/accuracy computed only over the final L timesteps (the output phase)."""
    out_logits = logits[:, -L:, :]
    loss = F.cross_entropy(out_logits.reshape(-1, out_logits.shape[-1]), targets.reshape(-1))
    preds = out_logits.argmax(dim=-1)
    acc = (preds == targets).float().mean().item()
    return loss, acc


def train_copy_model(cell_type, T, steps, batch_size=TRAIN_BATCH_SIZE, lr=2e-3, grad_clip=5.0, seed=0):
    torch.manual_seed(seed)
    model = CopyModel(VOCAB_SIZE, D_MODEL, HIDDEN_SIZE, cell_type=cell_type).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for _ in range(steps):
        x, y = make_copy_batch(batch_size, T, device=device)
        logits = model(x)
        loss, _ = copy_task_loss_and_acc(logits, y, COPY_LEN)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()

    with torch.no_grad():
        x_eval, y_eval = make_copy_batch(EVAL_BATCH_SIZE, T, device=device)
        logits_eval = model(x_eval)
        _, final_acc = copy_task_loss_and_acc(logits_eval, y_eval, COPY_LEN)
    return model, final_acc

In [ ]:
results = {}
for cell_type in ["rnn", "lstm"]:
    for T in DELAY_VALUES:
        _, acc = train_copy_model(cell_type, T, steps=TRAIN_STEPS)
        results[(cell_type, T)] = acc
        print(f"cell={cell_type:4s} T={T:4d}  final per-symbol accuracy={acc:.3f}")

## Part D: gradient-flow diagnostic

At the fixed long delay `GRAD_DIAG_T`, run one forward pass on a **freshly initialized, untrained** model
with `return_states=True` (which calls `retain_grad()` on every per-timestep hidden/cell state), compute the
copy-task loss, call `.backward()` once, and read off `||dL/dh_t||` (and `||dL/dc_t||` for LSTM) at every
timestep `t`. Early timesteps only receive gradient through the recurrence (their own readout isn't part of
the loss, since only the final `COPY_LEN` timesteps are), so this isolates exactly the "does gradient survive
`T` steps of backprop through time" question.

In [ ]:
def gradient_flow_diagnostic(cell_type, T, batch_size=64, seed=0):
    torch.manual_seed(seed)
    model = CopyModel(VOCAB_SIZE, D_MODEL, HIDDEN_SIZE, cell_type=cell_type).to(device)
    x, y = make_copy_batch(batch_size, T, device=device)

    logits, hidden_states, cell_states = model(x, return_states=True)
    loss, _ = copy_task_loss_and_acc(logits, y, COPY_LEN)
    model.zero_grad()
    loss.backward()

    h_norms = [h.grad.norm().item() if h.grad is not None else 0.0 for h in hidden_states]
    c_norms = [c.grad.norm().item() if c.grad is not None else 0.0 for c in cell_states] if cell_states else None
    return h_norms, c_norms


rnn_h_norms, _ = gradient_flow_diagnostic("rnn", GRAD_DIAG_T)
lstm_h_norms, lstm_c_norms = gradient_flow_diagnostic("lstm", GRAD_DIAG_T)
print("RNN  ||dL/dh_t||  first 5 / last 5:", [round(v, 6) for v in rnn_h_norms[:5]], [round(v, 6) for v in rnn_h_norms[-5:]])
print("LSTM ||dL/dh_t||  first 5 / last 5:", [round(v, 6) for v in lstm_h_norms[:5]], [round(v, 6) for v in lstm_h_norms[-5:]])

## Part E: plots

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
for cell_type in ["rnn", "lstm"]:
    accs = [results[(cell_type, T)] for T in DELAY_VALUES]
    plt.plot(DELAY_VALUES, accs, marker="o", label=cell_type)
plt.xlabel("delay length T")
plt.ylabel("final per-symbol copy accuracy")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.title("Copy-task accuracy vs. delay length: vanilla RNN vs. LSTM")
plt.show()

In [ ]:
plt.figure()
plt.plot(range(len(rnn_h_norms)), rnn_h_norms, label="RNN hidden state h_t")
plt.plot(range(len(lstm_h_norms)), lstm_h_norms, label="LSTM hidden state h_t")
plt.plot(range(len(lstm_c_norms)), lstm_c_norms, label="LSTM cell state c_t", linestyle="--")
plt.xlabel("timestep t")
plt.ylabel("||dL/d(state_t)||")
plt.yscale("log")
plt.legend()
plt.title(f"Gradient-flow diagnostic at T={GRAD_DIAG_T} (fresh, untrained models)")
plt.show()

## Discussion

**Expected qualitative pattern:** the vanilla RNN's copy accuracy should stay high for small `T` and then
collapse toward chance (`1/NUM_SYMBOLS`) past some critical delay, while the LSTM should stay close to
perfect across the whole sweep. In the gradient-flow plot, the vanilla RNN's `||dL/dh_t||` should decay
roughly exponentially (roughly a straight line on the log-scale y-axis) as `t` moves away from the loss, while
the LSTM's `||dL/dh_t||` and `||dL/dc_t||` should stay much flatter -- the additive cell-state update lets
gradient flow backward closer to unchanged at each step (scaled by the forget gate) instead of being
repeatedly multiplied through `W_hh` and `tanh'`, which is what drives the RNN's decay.

**With `SMOKE_TEST` scale (tiny delays, few training steps), don't expect a dramatic version of this effect**
-- delays of 2-4 steps are short enough that even the vanilla RNN can often bridge them, and 20 training
steps is not enough to reach a stable final accuracy either way. Re-run with `SMOKE_TEST` unset (the default
`DELAY_VALUES = [5, 20, 50, 100]` and `TRAIN_STEPS = 500`) to see the effect at the scale it's meant to be
observed at. Increasing `TRAIN_STEPS`, `HIDDEN_SIZE`, or adding delays beyond 100 will sharpen the contrast
further.